In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import matplotlib
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

matplotlib.rcParams['figure.figsize'] = (12, 8)

import warnings
warnings.filterwarnings('ignore')

## HW16 — метод k ближайших соседей

Сравните линейные модели и kNN. В классификации соседи голосуют, в регрессии их ответы усредняются; расстояние и веса можно менять. Полные данные не пересчитывались.

In [ ]:
circ = pd.read_csv('concertriccir2.csv')
circ.head()

In [ ]:
sns.scatterplot(data=circ, x='X', y='Y', hue='class')

### Задание 1

Обучите LogisticRegression на данных окружностей, изобразите разделяющую прямую и посчитайте accuracy на всей выборке без валидации. Объясните результат по геометрии.

In [ ]:
from sklearn.linear_model import LogisticRegression

### Your code is here
X = circ.drop('class', axis=1)
y = circ['class']

logreg = LogisticRegression()

logreg.fit(X, y)
logreg.score(X, y)


In [ ]:
from mlxtend.plotting import plot_decision_regions
plot_decision_regions(X=X.to_numpy(),y=y.to_numpy().astype(int),clf=logreg)

### Задание 2

Разделите данные, обучите шесть kNN для all_k и сохраните модели в список. Посчитайте accuracy на train/test и сравните число соседей. Сохранение списка в исходнике не реализовано.

In [ ]:
X = circ[['X', 'Y']].values
y = circ['class'].values.astype('int')

In [ ]:
from sklearn.model_selection import train_test_split, GridSearchCV

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                                    shuffle=True, random_state=2022)

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
all_k = [1, 3, 10, 30, 40, 60]

### Your code is here
train_accuracies = []
test_accuracies = []
for k in all_k:

    knn = KNeighborsClassifier(n_neighbors=k)

    knn.fit(X_train, y_train)
    train_accuracies.append(knn.score(X_train, y_train))
    test_accuracies.append(knn.score(X_test, y_test))
    print(f"train_accuracies: {k} - {train_accuracies[-1]}, test_accuracies: {k} - {test_accuracies[-1]}")
    print("====\n")


### Бонус: решающие поверхности

С помощью mlxtend.plot_decision_regions изобразите поверхности шести моделей. Для размещения графиков можно использовать gridspec.GridSpec и itertools.product. Ячейка решения оставлена пустой.

In [ ]:
import mlxtend
from mlxtend.plotting import plot_decision_regions
import matplotlib.gridspec as gridspec
import itertools

In [ ]:
### Your code is here




Как меняется поверхность при росте k? Какой k подходит геометрии данных? Сравните kNN с линейной моделью.

### Задание 4

На Sigma Cabs сравните kNN с нормированием и без него для k из [1,5,10]. Посчитайте качество на test и объясните роль нормирования.

In [ ]:
df = pd.read_csv('processed_sigma_cabs.csv')
df.head()

In [ ]:
X = df.drop('Surge_Pricing_Type', axis=1)
y = df['Surge_Pricing_Type']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                                    shuffle=True, random_state=2022)

In [ ]:
### Your code is here
# param_grid = {
#     'knn__n_neighbors': [1,5,10]
# }
# knn._parameter_constraints
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

test_accuracies = []
for k in [1,5,10]:
    pipe = Pipeline([('scaler',StandardScaler()),
                         ('knn',KNeighborsClassifier(n_neighbors=k))])
    pipe.fit(X_train, y_train)
    test_accuracies.append(pipe.score(X_test, y_test))
    print(f"test_accuracies: {k} - {test_accuracies[-1]}")
    print("====\n")

In [ ]:
test_accuracies = []
for k in [1,5,10]:
    pipe = Pipeline([('knn',KNeighborsClassifier(n_neighbors=k))])
    pipe.fit(X_train, y_train)
    test_accuracies.append(pipe.score(X_test, y_test))
    print(f"test_accuracies: {k} - {test_accuracies[-1]}")
    print("====\n")

### Задание 5

Для нормированных данных используйте расстояние Минковского (sum(|x-z|**p))**(1/p) с p=1. Сравните результат с предыдущим p=2.

In [ ]:
### Your code is here

pipe = Pipeline([('scaler',StandardScaler()),
            ('knn',KNeighborsClassifier(n_neighbors=10,metric='minkowski',p=1))])

pipe.fit(X_train, y_train)
pipe.score(X_test, y_test)



### Задание 6: регрессия стоимости жилья

На California Housing используйте longitude и latitude. Изобразите объекты и объясните смысл соседства. Сравните RMSE линейной регрессии на train/test; подберите k из 1..9 для нормированного kNN-регрессора. Для гауссовских весов возьмите выбранный k и подберите h из np.arange(0.02,0.1,0.01). Подбор h в исходнике не реализован.

In [ ]:
df = pd.read_csv('housing.csv')
df.head()

In [ ]:
X = df[['longitude', 'latitude']]
y = df['median_house_value']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                                    shuffle=True, random_state=2022)
sns.scatterplot(data=X, x='longitude', y='latitude', hue=y);

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error

linear_model = LinearRegression()

linear_model.fit(X_train, y_train)
# y_pred_train = linear_model.score(X_train, y_train)
y_pred_train =linear_model.predict(X_train)
y_pred_test =linear_model.predict(X_test)
print(f"train RMSE: {root_mean_squared_error(y_train, y_pred_train)} test RMSE: {root_mean_squared_error(y_test, y_pred_test)}")

In [ ]:
### Your code is here
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsRegressor

param_grid = {
    'knn__n_neighbors':[1,2,3,4,5,6,7,8,9]
}

pipe = Pipeline([('scaler',StandardScaler()),
            ('knn',KNeighborsRegressor())])

grid = GridSearchCV(estimator=pipe, param_grid=param_grid, scoring='neg_root_mean_squared_error')

grid.fit(X_train, y_train)
root_mean_squared_error(y_test, grid.predict(X_test))
# root_mean_squared_error(y_train, grid.predict(X_train))

# best_k = grid.best_params_

In [ ]:
# [0.02,0.1,0.01]
def kernel(distances, h=100):

        const = 1 / (np.sqrt(2 * np.pi))
        power = (-1/2) * ((distances)**2) / h**2

        return const * np.exp(power)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                                    shuffle=True, random_state=2022)

pipe = Pipeline([('scaler', StandardScaler()),
                 ('KNN', KNeighborsRegressor(n_neighbors=grid.best_params_['knn__n_neighbors'], weights=kernel))])

pipe.fit(X_train, y_train)
print(f"train RMSE: {root_mean_squared_error(y_train, pipe.predict(X_train))} test RMSE: {root_mean_squared_error(y_test, pipe.predict(X_test))}")


In [ ]:
print(f"RMSE на трейне: {root_mean_squared_error(y_train, pipe.predict(X_train)).round(3)}")

print(f"RMSE на тесте: {root_mean_squared_error(y_test, pipe.predict(X_test)).round(3)}")
# Качество на трейне: 0.632
# Качество на тесте: 0.035

# Качество на трейне: 0.627
# Качество на тесте: 0.035

